# Terrain profiles at the 8 RCT bridge sites

Question: can the free Copernicus 30 m DEM reproduce the spans Fika actually built?

Workflow per site: look at the elevation heatmap, sweep profile bearings to find the one that crosses the valley, then estimate bank-to-bank width and compare to the as-built `Span (m)`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from b2p.dem import fetch_clip, profile_through

df = pd.read_csv('../data/rct_2023_bridges.csv', encoding='utf-8-sig')
df[['Bridge Name', 'GPS (Latitude)', 'GPS (Longitude)', 'Span (m)']]

In [ ]:
# Pick a site and look at its terrain
site = df.iloc[0]
lat, lon = site['GPS (Latitude)'], site['GPS (Longitude)']
data, transform = fetch_clip(lat, lon)

plt.figure(figsize=(6, 6))
plt.imshow(data, cmap='terrain')
plt.colorbar(label='elevation (m)')
plt.scatter([data.shape[1] / 2], [data.shape[0] / 2], c='red', marker='x')
plt.title(f"{site['Bridge Name']} (span {site['Span (m)']} m)")
plt.show()

In [ ]:
# Sweep bearings to find the profile that crosses the valley.
# The right bearing shows a V or U shape centered near distance 0.
fig, axes = plt.subplots(2, 3, figsize=(14, 7), sharey=True)
for ax, az in zip(axes.flat, range(0, 180, 30)):
    dist, elev = profile_through(data, transform, lat, lon, azimuth_deg=az)
    ax.plot(dist, elev)
    ax.axvline(0, color='red', ls=':')
    ax.set_title(f'bearing {az}\u00b0')
fig.suptitle(site['Bridge Name'])
fig.supxlabel('distance from site (m)')
fig.supylabel('elevation (m)')
plt.tight_layout()
plt.show()

## Next steps

- For each site, record the valley-crossing bearing and estimate bank-to-bank width from the profile.
- Scatter estimated width vs. as-built `Span (m)` across all 8 sites.
- Note where 30 m resolution breaks down (short spans like Mukamira at 38 m have barely one pixel across the gap). That finding feeds the DEM-resolution conversation on Thursday.

In [ ]:
data, transform = fetch_clip(lat, lon, half_size_m=1500)

fig, axes = plt.subplots(1, 5, figsize=(18, 4), sharey=True)
for ax, az in zip(axes, [40, 50, 60, 70, 80]):
    dist, elev = profile_through(
        data, transform, lat, lon, azimuth_deg=az, length_m=2800, n_points=1400
    )
    ax.plot(dist, elev)
    ax.axvline(0, color='red', ls=':')
    ax.set_title(f'bearing {az}\u00b0')
fig.supxlabel('distance from site (m)')
fig.supylabel('elevation (m)')
plt.tight_layout()
plt.show()

In [ ]:
spans 